# ⚡ Proposed Model: Dual-Stream Binary Engagement Transformer

### **Overview:**
Binary student engagement classification (**Engaged vs Disengaged**) using **EfficientNet-B0 + OpenFace Dual-Stream Feature Fusion** followed by a **3-Layer Temporal Transformer Encoder Stack**.

| Component | Architecture Specification |
|---|---|
| **Stream 1 (Visual CNN)** | EfficientNet-B0 (1280-dim per frame) |
| **Stream 2 (Facial AUs)** | OpenFace (709-dim Action Units, landmarks, gaze, head pose) |
| **Dual-Stream Fusion** | Concatenation (1989-dim) $\rightarrow$ Linear Projection (128-dim) + LayerNorm |
| **Temporal Modeling** | ⚡ **3-Layer Multi-Head Self-Attention Transformer Encoder** (Pre-LN, 4 Heads) |
| **Temporal Pooling** | Masked Temporal Mean Pooling |
| **Classification Head** | 3-Layer MLP Head (128 $\rightarrow$ 64 $\rightarrow$ 32 $\rightarrow$ 2) |
| **Target Classes** | 2 (Engaged: `0`, Disengaged: `1`) |
| **Training Duration** | Up to 60 Epochs (AdamW, Cosine Annealing, Early Stopping with `patience=10`) |

---

### **Step 1: Mount Google Drive**
Click the **Play ▶** button. When prompted, authorize access to your Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive successfully mounted!")

### **Step 2: Set Working Directory**
Navigates to `colab_binary_transformer` in your Google Drive.

In [ ]:
import os
import sys

possible_paths = [
    "/content/drive/MyDrive/Student_Engagement_DL/colab_binary_transformer",
    "/content/drive/MyDrive/colab_binary_transformer",
    "/content/drive/MyDrive/Binary classification/colab_binary_transformer",
    "/content/drive/Student_Engagement_DL/colab_binary_transformer",
]

working_dir = None
for p in possible_paths:
    if os.path.exists(p):
        working_dir = p
        break

if working_dir is None:
    working_dir = "/content/drive/MyDrive/Student_Engagement_DL/colab_binary_transformer"
    os.makedirs(working_dir, exist_ok=True)

if working_dir not in sys.path:
    sys.path.insert(0, working_dir)

%cd $working_dir
print("\n📁 Current working directory:", os.getcwd())
print("📄 Available files:", [f for f in os.listdir('.') if not f.startswith('.')])

### **Step 3: Install Required Dependencies**
Installs PyTorch, torchvision, scikit-learn, pandas, matplotlib, and seaborn.

In [ ]:
!pip install -q torch torchvision scikit-learn pandas matplotlib seaborn

### **Step 4: Verify Input Data (`sequence_fuse_features_binary`)**
Confirms that your preprocessed feature files are detected and checks their dimension (`1989 = 1280 + 709`).

In [ ]:
import config_binary_transformer as config
import torch

seq_dir = config.SEQUENCE_FUSED_DIR
print(f"Checking feature folder: {seq_dir}")

if os.path.exists(seq_dir):
    files = [f for f in os.listdir(seq_dir) if f.endswith('.pt')]
    print(f"✅ Successfully found {len(files)} preprocessed feature files (.pt)!")
    if len(files) > 0:
        sample = torch.load(os.path.join(seq_dir, files[0]), map_location="cpu")
        print(f"   - Sample video     : {files[0]}")
        print(f"   - Frame count      : {sample['sequence'].shape[0]}")
        print(f"   - Feature dimension: {sample['sequence'].shape[1]} (1280 CNN + 709 OpenFace)")
        print(f"   - Class label      : {sample.get('label', 'N/A')} ({config.CLASS_NAMES[sample.get('label', 0)]})")
else:
    print(f"❌ Folder not found at {seq_dir}!")

### **Step 5: Inspect Transformer Architecture & Parameter Count**
Builds the `BinaryEngagementTransformer` and calculates trainable parameters.

In [ ]:
from models_binary_transformer import BinaryEngagementTransformer
import config_binary_transformer as config

model = BinaryEngagementTransformer(
    input_dim=config.D_MODEL if False else 1989,
    d_model=config.D_MODEL,
    nhead=config.NHEAD,
    num_layers=config.NUM_LAYERS,
    dim_feedforward=config.DIM_FEEDFORWARD,
    dropout=config.DROPOUT,
    num_classes=config.NUM_CLASSES
)

total_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("=" * 65)
print("  BINARY ENGAGEMENT TRANSFORMER — PARAMETER SUMMARY")
print("=" * 65)
print(f"  Dual-Stream Input Dimension : 1989 (1280 CNN + 709 OpenFace)")
print(f"  Transformer d_model         : {config.D_MODEL}")
print(f"  Multi-Head Attention Heads  : {config.NHEAD}")
print(f"  Transformer Encoder Layers  : {config.NUM_LAYERS}")
print(f"  FeedForward Expansion Dim   : {config.DIM_FEEDFORWARD}")
print(f"  Total Trainable Parameters  : {total_trainable:,}")
print("=" * 65)

### **Step 6: Train the Binary Transformer**
Trains the Transformer on sliding windows using AdamW, Cosine Annealing, and Early Stopping (`patience=10`). Displays loss and accuracy per epoch.

In [ ]:
!python train_binary_transformer.py

### **Step 7: Evaluate on Held-Out Test Videos**
Runs video-level inference with soft-voting aggregation across all windows per video. Computes Accuracy, Precision, Recall, F1-Score, Confusion Matrix, and ROC Curve.

In [ ]:
!python evaluate_binary_transformer.py

### **Step 8: Visualize Evaluation Plots & Confusion Matrix**
Displays the Confusion Matrix and ROC Curve plots directly in the notebook.

In [ ]:
from IPython.display import Image, display
import config_binary_transformer as config

print("📊 1. Confusion Matrix (Binary Transformer):")
if os.path.exists(config.CONFUSION_MATRIX_PATH):
    display(Image(filename=config.CONFUSION_MATRIX_PATH))

print("\n📈 2. ROC Curve (Binary Transformer):")
if os.path.exists(config.ROC_CURVE_PATH):
    display(Image(filename=config.ROC_CURVE_PATH))

### **Step 9: View Video Predictions Summary**
Displays the first 15 test video predictions with confidence probabilities.

In [ ]:
import pandas as pd
import config_binary_transformer as config

if os.path.exists(config.PREDICTIONS_CSV_PATH):
    df = pd.read_csv(config.PREDICTIONS_CSV_PATH)
    display(df.head(15))
else:
    print("Predictions file not found. Run Step 7 first.")